# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Machine Learning: Alternating Least Squares (ALS)** </center>
---
**Profesor**: Pablo Camarillo Ramirez
---
***Student***: Nicolas Navarro Valenzuela

# Create SparkSession

In [1]:
from spark_utils import SparkUtils

su = SparkUtils("ML: ALS", 
                "spark://spark-master:7077")
su.spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/23 00:46:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Example 1: Songs recommednation

In [2]:
# Sample user-song interaction data
data = [(1, 1, 4),
        (1, 2, 5),
        (1, 5, 5),
        (2, 2, 3),
        (2, 3, 4),
        (2, 4, 3),
        (3, 1, 2),
        (3, 3, 5),
        (3, 5, 1)]
  
# Define schema for the DataFrame
schema = SparkUtils.generate_schema([("user_id", "int"), ("song_id", "int"), ("rating", "int")])

# Create DataFrame for interactions
interactions_df = su.spark.createDataFrame(data, schema)
interactions_df.show()

+-------+-------+------+
|user_id|song_id|rating|
+-------+-------+------+
|      1|      1|     4|
|      1|      2|     5|
|      1|      5|     5|
|      2|      2|     3|
|      2|      3|     4|
|      2|      4|     3|
|      3|      1|     2|
|      3|      3|     5|
|      3|      5|     1|
+-------+-------+------+



In [8]:
print(f"Number of items o canciones (n):{interactions_df.groupBy('song_id').count().count()}")
print(f"Number of users (m):{interactions_df.groupBy('user_id').count().count()}")

Number of items o canciones (n):5
Number of users (m):3


In [ ]:
from pyspark.ml.recommendation import ALS
als = ALS(
    userCol="user_id", 
    itemCol="song_id", 
    ratingCol="rating", 
    maxIter=10, 
    regParam=0.1, 
    rank=5, # Controls the dimensionality of the latent vector space for 
            # users and items.
    coldStartStrategy="drop"  # Avoids NaN predictions
)

In [10]:
model = als.fit(interactions_df)
print("Recommendation system generated successfully")

Recommendation system generated successfully


In [11]:
# Generate recommendations for each user
user_recommendations = model.recommendForAllUsers(numItems=3)

# Show recommendations
user_recommendations.show(truncate=False)

+-------+------------------------------------------------+
|user_id|recommendations                                 |
+-------+------------------------------------------------+
|1      |[{2, 4.9345202}, {5, 4.8657546}, {1, 3.9491198}]|
|2      |[{3, 3.956777}, {2, 2.9636896}, {4, 2.903076}]  |
|3      |[{3, 4.8436656}, {4, 2.8235016}, {1, 1.970727}] |
+-------+------------------------------------------------+



In [12]:
songs = [
    (1, "song a"),
    (2, "song b"),
    (3, "song c"),
    (4, "song d"),
    (5, "song e")]

songs_schema = SparkUtils.generate_schema([("song_id", "int"), ("title", "string")])
songs_df = su.spark.createDataFrame(songs, songs_schema)

In [13]:
from pyspark.sql.functions import explode

# Explode recommendations for easier reading
recommendations = user_recommendations.select("user_id", explode("recommendations").alias("rec"))
recommendations = recommendations.join(songs_df, recommendations.rec.song_id == songs_df.song_id).select("user_id", "title", "rec.rating")

# Show user-song recommendations with titles
recommendations.show(truncate=False)

+-------+------+---------+
|user_id|title |rating   |
+-------+------+---------+
|1      |song b|4.9345202|
|1      |song e|4.8657546|
|1      |song a|3.9491198|
|2      |song c|3.956777 |
|2      |song b|2.9636896|
|2      |song d|2.903076 |
|3      |song c|4.8436656|
|3      |song d|2.8235016|
|3      |song a|1.970727 |
+-------+------+---------+



In [14]:
predictions = model.transform(interactions_df)
predictions.show(truncate=False)

+-------+-------+------+----------+
|user_id|song_id|rating|prediction|
+-------+-------+------+----------+
|1      |1      |4     |3.9491198 |
|1      |2      |5     |4.9345202 |
|1      |5      |5     |4.8657546 |
|2      |2      |3     |2.9636896 |
|3      |1      |2     |1.970727  |
|3      |3      |5     |4.8436656 |
|3      |5      |1     |1.0578728 |
|2      |3      |4     |3.956777  |
|2      |4      |3     |2.903076  |
+-------+-------+------+----------+



In [15]:
# Evaluate the Recommendation System
from pyspark.ml.evaluation import RegressionEvaluator
# Set up evaluator to compute RMSE
evaluator = RegressionEvaluator(
    metricName="rmse", 
    labelCol="rating", 
    predictionCol="prediction"
)

# Calculate RMSE
rmse = evaluator.evaluate(predictions)
print(f"Root-mean-square error (RMSE) = {rmse}")

Root-mean-square error (RMSE) = 0.08571644041250898


# Lab 12: Building a Recommendation System with ALS 

In [16]:
movies_ratings_path = "/opt/spark/work-dir/data/ml/als"

movies_ratings_schema = SparkUtils.generate_schema([("user_id", "int"), ("movie_id", "int"), ("rating", "int"),("timestamp", "int")])

# Source https://github.com/databricks/Spark-The-Definitive-Guide/blob/master/data/sample_movielens_ratings.txt
movies_ratings_df = su.spark.read \
                    .option("header", "false") \
                    .option("delimiter", "::") \
                    .schema(movies_ratings_schema) \
                    .csv(movies_ratings_path)

movies_ratings_df.printSchema()
movies_ratings_df.show(n=3)

root
 |-- user_id: integer (nullable = true)
 |-- movie_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- timestamp: integer (nullable = true)



+-------+--------+------+----------+
|user_id|movie_id|rating| timestamp|
+-------+--------+------+----------+
|      0|       2|     3|1424380312|
|      0|       3|     1|1424380312|
|      0|       5|     2|1424380312|
+-------+--------+------+----------+
only showing top 3 rows


## Create & Train the ML Model

In [60]:
# Create and train the ALS model
from pyspark.ml.recommendation import ALS

als_movies = ALS(
    userCol="user_id", 
    itemCol="movie_id", 
    ratingCol="rating", 
    maxIter=20, 
    regParam=0.001, 
    rank=5, 
    coldStartStrategy="drop"
)

model_movies = als_movies.fit(movies_ratings_df)
print("Movie recommendation system generated successfully")

Movie recommendation system generated successfully


## Persist the model

## Predictions

In [61]:
# Recomendations and prediction for all users
user_recommendations_movies = model_movies.recommendForAllUsers(numItems=3)
user_recommendations_movies.show(n=10, truncate=False)

+-------+---------------------------------------------------+
|user_id|recommendations                                    |
+-------+---------------------------------------------------+
|0      |[{93, 2.8139758}, {92, 2.7092118}, {2, 2.4720387}] |
|10     |[{93, 3.7814474}, {46, 3.2637086}, {12, 3.1618586}]|
|20     |[{22, 4.1351767}, {28, 3.9884865}, {68, 3.7596197}]|
|1      |[{22, 3.2893698}, {68, 3.0958}, {28, 3.0406144}]   |
|11     |[{46, 6.4539013}, {34, 6.2755966}, {74, 5.8460503}]|
|21     |[{41, 5.000789}, {76, 4.994286}, {70, 4.918502}]   |
|22     |[{28, 5.799618}, {53, 5.591397}, {59, 5.448506}]   |
|2      |[{93, 5.177916}, {8, 4.7592263}, {83, 4.5240164}]  |
|12     |[{46, 8.500153}, {55, 6.301381}, {49, 5.8496943}]  |
|23     |[{46, 7.293328}, {55, 5.592705}, {90, 5.233742}]   |
+-------+---------------------------------------------------+
only showing top 10 rows


In [62]:
predictions_movies = model_movies.transform(movies_ratings_df)
predictions_movies.show(n=10, truncate=False)

+-------+--------+------+----------+----------+
|user_id|movie_id|rating|timestamp |prediction|
+-------+--------+------+----------+----------+
|22     |0       |1     |1424380312|0.9477714 |
|22     |3       |2     |1424380312|1.7503326 |
|22     |5       |2     |1424380312|2.0036032 |
|22     |6       |2     |1424380312|2.4222383 |
|22     |9       |1     |1424380312|1.5616082 |
|22     |10      |1     |1424380312|1.5658782 |
|22     |11      |1     |1424380312|1.1256938 |
|22     |13      |1     |1424380312|1.4921011 |
|22     |14      |1     |1424380312|1.4147668 |
|22     |16      |1     |1424380312|0.84936637|
+-------+--------+------+----------+----------+
only showing top 10 rows


## Test ML Model

In [63]:
movies_evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse_movies = movies_evaluator.evaluate(predictions_movies)
print(f"Root-mean-square error (RMSE) for movie recommendations = {rmse_movies}")

Root-mean-square error (RMSE) for movie recommendations = 0.5040116224949202


In [64]:
su.spark.stop()